# 101 — Re-ranking y filtros de evidencia

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Iter 1: gana d1. Iter 2 con λ = 0.5:
`MMR(d2) = 0.5·0.85 − 0.5·0.95 = −0.050`, `MMR(d3) = 0.5·0.70 − 0.5·0.30 = 0.200`,
`MMR(d4) = 0.5·0.60 − 0.5·0.20 = 0.200`. Empate d3/d4 (se resuelve por score base:
d3). La selección {d1, d3} no cambia, pero d2 pasa de 0.310 a valor negativo: con λ
bajo, la redundancia domina.

**Ejercicio 2.** (a) `100 · 8 ms = 0.8 s` de re-ranking + 30 ms de recuperación.
(b) `500 000 · 8 ms = 4 000 s ≈ 1.11 horas` por consulta. Por eso el cross-encoder
nunca es primera etapa.

**Ejercicio 3.** Con umbral 0.55 pasan 2 documentos (0.91 y 0.88): el salto de 0.88 a
0.52 es la frontera natural de evidencia. Si un umbral estricto deja menos fuentes de
las que exige la pregunta, el sistema debe declararlo ("evidencia insuficiente") o
relajar el umbral de forma explícita y registrada — nunca rellenar en silencio con
documentos débiles.

**Ejercicio 4.** El contrato se verifica en el código: `kind == "retrieval"` y
`evidence` no vacía. El detalle interno puede variar con la semilla; el contrato no.

In [ ]:
result = run_lab("retrieval", seed=101)
assert result["kind"] == "retrieval"
assert result["evidence"]
show(result)


In [ ]:
lam = 0.5
score = {"d1": 0.90, "d2": 0.85, "d3": 0.70, "d4": 0.60}
sim = {("d1", "d2"): 0.95, ("d1", "d3"): 0.30, ("d1", "d4"): 0.20,
       ("d2", "d3"): 0.35, ("d2", "d4"): 0.25, ("d3", "d4"): 0.15}

def s(a, b):
    return sim.get((a, b)) or sim.get((b, a)) or 0.0

def mmr(d, sel):
    pen = max(s(d, x) for x in sel) if sel else 0.0
    return lam * score[d] - (1 - lam) * pen

sel = []
candidatos = list(score)
for _ in range(2):
    mejor = max((d for d in candidatos if d not in sel),
                key=lambda d: (mmr(d, sel), score[d]))
    print({d: round(mmr(d, sel), 3) for d in candidatos if d not in sel}, "->", mejor)
    sel.append(mejor)
print("Selección:", sel)

# Ejercicio 2
print("re-rank top-100:", 100 * 8 / 1000, "s")
print("colección entera:", 500_000 * 8 / 1000 / 3600, "horas")

# Ejercicio 3
scores = [0.91, 0.88, 0.52, 0.49, 0.11]
print("pasan umbral 0.55:", [x for x in scores if x >= 0.55])

## Reflexión

1. Si el recall@100 de la primera etapa es 0.60, ¿qué fracción máxima de respuestas correctas puede lograr el pipeline aunque el cross-encoder fuera perfecto, y por qué?
2. En el ejemplo MMR, ¿con qué valor aproximado de λ volvería d2 a entrar en la selección? Plantea la desigualdad `λ·0.85 − (1−λ)·0.95 > λ·0.70 − (1−λ)·0.30`.
3. ¿Por qué un umbral de score de cross-encoder calibrado en MS MARCO no es transferible a tu corpus interno, y qué datos necesitarías para recalibrarlo?